# Seeds and Augmentation — putting error bars on the headline claim

**This is the final run. Version 4 answered three questions and left one confounded; this
run closes it.** What v4 established, on its corrected (clipped, floor-relative) metric:

* **The sweep winner's architecture is not real.** `winner` beat `v12` by +0.376 dB against a
  combined spread of 1.344 — indistinguishable from seed noise. The +4.16 dB was a single draw.
* **Training duration is the one solid lever.** `epochs_run` vs PSNR came out at r=+0.721 over
  12 runs, replicating the sweep's +0.650. Going from `patience=5` to `patience=10` bought
  +21 epochs and +1.298 dB *and halved the seed spread* (0.896 → 0.484).
* **`max_epochs=60` was binding.** `v12` seed 42 and `winner_p10` seed 44 both ran 60/60 and
  were the best run in their arm — those models were still improving when the ceiling cut
  them off. The top was never found.
* **Augmentation looked best on every axis** (+1.322 dB, best M0/M1/M2, invented-blob rate
  37.7% → 29.7%, tightest spread) — **but it is confounded with duration.** `winner_aug` ran
  a mean 38.7 epochs against plain `winner`'s 27.3: augmentation delayed overfitting, so early
  stopping fired later, so it trained longer. v4 cannot separate regularisation from epochs.

**Design.** Three configurations x 5 seeds = 15 training runs, on a schedule long enough that
no arm is truncated. Once duration is no longer a free variable, the remaining gap between an
augmented and an un-augmented arm *is* the augmentation effect. Only the training seed varies
within an arm; the cube split is held fixed, so the spread measured is training variance.

| Config | What it tests |
|---|---|
| `base` — base 32, 1x2x4, alpha 0.8, batch 32 | the reference, converged, with an error bar |
| `base_aug` — `base` + D4 augmentation | **does augmentation help once duration is matched?** |
| `wide_aug` — base 48, 1x2x4x8, alpha 0.888, lr 8.2e-4 + D4 | is the wider architecture real at n=5? |

`base` vs `base_aug` isolates augmentation. `base_aug` vs `wide_aug` isolates architecture —
both augmented, both on the same schedule. v4 could do neither: all three of its promising arms
were built on `winner`, an architecture that earned nothing.

n was raised from 3 to 5 because nothing in v4 cleared the promotion bar. `verdict()` calls a
gap REAL only above 2x the combined spread — roughly 2 dB at v4's spreads — so every result
stalled at "suggestive".

Then the best seed of each goes through the **all-5-holdout moment-map protocol**, because the
beam A/B already demonstrated that a pixel-metric win can coexist with a *worse* scientific
deliverable (M0 fell from +69.8% to +59.8% while PSNR rose).

**Cost.** From v4's measured throughput — 39.7 s/epoch for the 32-channel arms (batch 32) and
94.3 s/epoch for the 48-channel arm (batch 16) — 15 runs at an expected ~80 epochs is roughly
**19 h GPU on T4x2**, plus ~10 min per moment-map evaluation. That is 2–3 Kaggle sessions.
Section 1b restores completed arms from the previous version's Output, and `repeat_config`
appends a CSV row and saves a checkpoint the moment each seed finishes, so a timeout costs at
most one run. Requires GPU. Run `07-classical-baselines.ipynb` in a separate CPU-only session
in parallel — it needs no GPU quota.

## 0. Bootstrap

In [1]:
import os, sys, subprocess, glob

ON_KAGGLE = os.path.exists('/kaggle')
BRANCH = 'midterm-prep'
if ON_KAGGLE:
    REPO='/kaggle/working/EXXA'; PKG=os.path.join(REPO,'DENOISING_DIFFUSION')
    if not os.path.exists(REPO):
        subprocess.run(['git','clone','--branch',BRANCH,'--depth','1',
                        'https://github.com/KrishanYadav333/EXXA.git',REPO], check=True)
    else:
        subprocess.run(['git','-C',REPO,'fetch','origin',BRANCH], check=True)
        subprocess.run(['git','-C',REPO,'reset','--hard','origin/'+BRANCH], check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps',
                    'pytorch-msssim','bettermoments'], check=True)
    os.chdir(os.path.join(PKG,'notebooks'));  sys.path.insert(0, PKG)
    hits = glob.glob('/kaggle/input/**/*_dirty.fits', recursive=True)
    DATA_DIR = os.path.dirname(os.path.dirname(hits[0])) if hits else None
else:
    if os.path.basename(os.getcwd()) != 'notebooks' and os.path.isdir('notebooks'): os.chdir('notebooks')
    sys.path.insert(0, os.path.abspath('..'))
    DATA_DIR = '../data/Line Emission Data'
print('cwd:', os.getcwd(), '| DATA_DIR:', DATA_DIR)

Cloning into '/kaggle/working/EXXA'...
Updating files: 100% (5243/5243), done.


cwd: /kaggle/working/EXXA/DENOISING_DIFFUSION/notebooks | DATA_DIR: /kaggle/input/datasets/krishanyadav333/line-emission-data/Line Emission Data


## 0b. Pull latest code (re-run anytime — no kernel restart)

In [2]:
import os, sys, subprocess
ON_KAGGLE = os.path.exists('/kaggle'); REPO = '/kaggle/working/EXXA'; BRANCH = 'midterm-prep'
if ON_KAGGLE and os.path.exists(REPO):
    subprocess.run(['git','-C',REPO,'fetch','origin',BRANCH], check=True)
    subprocess.run(['git','-C',REPO,'reset','--hard','origin/'+BRANCH], check=True)
    print(subprocess.run(['git','-C',REPO,'log','--oneline','-1'],
                         capture_output=True, text=True).stdout.strip())
for _m in [m for m in list(sys.modules) if m == 'src' or m.startswith('src.')]:
    del sys.modules[_m]
print('src.* cleared — re-run the imports cell below.')

From https://github.com/KrishanYadav333/EXXA
 * branch            midterm-prep -> FETCH_HEAD


HEAD is now at 695202b feat(nb08): final run design -- 3 arms x 5 seeds at a converged schedule
695202b feat(nb08): final run design -- 3 arms x 5 seeds at a converged schedule
src.* cleared — re-run the imports cell below.


## 1. Imports, device, config

In [3]:
import csv, math, time
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from src.data.cube_split import split_cubes
from src.data.fits_cube_dataset import FITSChannelDataset, continuum_of
from src.models.unet import UNet
from src.training.sweep import repeat_config

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_GPU = torch.cuda.device_count()
SEED  = 42
np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print('device:', device, '| GPUs:', N_GPU,
      '->', [torch.cuda.get_device_name(i) for i in range(N_GPU)] if N_GPU else 'cpu')

TARGET_SIZE, N_SAMPLES = 256, 150
SUBTRACT_CONTINUUM, CONTINUUM_N = True, 5
N_SEEDS = 5                     # v4 ran 3 and nothing cleared the 2x-spread bar: every result
                                # stalled at "suggestive". 5 tightens the estimate enough for
                                # the ~1.3 dB effects v4 saw to be called either way.
NW = 2 if ON_KAGGLE else 0
# RULES.md #1 -- persist the moment training finishes, never in a cleanup cell.
# `../results/checkpoints` is INSIDE the git clone that the section-0 bootstrap wipes with
# `git reset --hard`, and it is not part of the notebook Output: a checkpoint written there
# survives only until the container dies. The top level of /kaggle/working is the only path
# that survives and the only one that becomes the Output.
#
# `repeat_config` saves each seed's best checkpoint and appends its CSV row the moment that
# seed finishes, so pointing CKPT_DIR straight at /kaggle/working IS the immediate persist --
# and it is stronger than a persist_ckpt() call, because no cell is left that can be skipped.
# Section 11's bulk copy stays on as a backstop. This run is 15 trainings over 2-3 sessions;
# a timeout must cost the seed in flight, not the session.
OUT_DIR   = '../results'
PERSIST   = '/kaggle/working' if ON_KAGGLE else OUT_DIR
CKPT_DIR  = PERSIST if ON_KAGGLE else '../results/checkpoints'
os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(CKPT_DIR, exist_ok=True)
# defined here, not at the training cell, because the restore (1b) and CSV-rebuild
# (3b) cells both read it and run first
REPEAT_CSV = os.path.join(PERSIST, 'seed_repeats.csv')   # sole record of the ranking

# Published reference, carried for context only -- RULES.md #6. These moments are on the
# UNCLIPPED whole-map metric (05 v12, before bab16d0), so they are NOT comparable to this
# notebook's M0/M1/M2 and every table below labels them as such. The PSNR is not comparable
# either: it was measured at N_SAMPLES=50 against this notebook's 150.
V12 = {'psnr': 32.95, 'ssim': 0.9857, 'mse': 0.000681,
       'M0': (69.8, 15.2), 'M1': (17.5, 7.8), 'M2': (20.1, 14.3)}
print(f'{N_SEEDS} seeds per config (configs defined in section 4)')

device: cuda | GPUs: 2 -> ['Tesla T4', 'Tesla T4']
5 seeds per config (configs defined in section 4)


## 1b. Restore a previous session (resume after a timeout)

The full run is several GPU-hours and a Kaggle session can be cut off part-way. Everything
needed to resume is already written as it goes: `repeat_config` appends one CSV row per seed the
moment it finishes, and saves that seed's best checkpoint.

They are written **straight to the top level of `/kaggle/working`** — `CKPT_DIR` and
`REPEAT_CSV` point there directly (section 1), so each checkpoint and CSV row is part of the
notebook *Output* the instant it is written. That is RULES.md #1: nothing depends on a later
cell being reached. Earlier versions wrote into `/kaggle/working/EXXA`, which the bootstrap
wipes with `git reset --hard` and which Kaggle does not carry between sessions, and copied to
the top level only in section 11 — so a timeout before that cell lost every model *and* the
Output this resume path reads from. Section 11 is now a backstop, not the save.

Tags matter here. Resume keys off `(tag, seed)`, so attaching an Output whose CSV uses a tag
this run also uses would skip that seed and reuse a model trained under different settings.
This run's tags — `base`, `base_aug`, `wide_aug` — collide with none of v4's, so attaching v4's
Output by mistake restores files that are simply never matched.

**To resume:** `Add Input` → `Notebook Output` → the previous version of this notebook. Then run
this cell. It finds `seed_repeats.csv` and any `*_seed*.pth` under `/kaggle/input/` and copies
them back into `../results`, so completed arms are reused instead of retrained. Safe to run
when there is nothing to restore — it just reports zero files.

In [4]:
import glob, shutil

RESTORED = {'csv': 0, 'ckpt': 0}
if ON_KAGGLE:
    # newest match wins, so re-running across several sessions picks the latest state
    csv_hits = sorted(glob.glob('/kaggle/input/**/seed_repeats.csv', recursive=True),
                      key=os.path.getmtime)
    if csv_hits:
        shutil.copy2(csv_hits[-1], REPEAT_CSV)
        RESTORED['csv'] = 1
        print('restored seed_repeats.csv from', csv_hits[-1])

    for src_p in glob.glob('/kaggle/input/**/*_seed*.pth', recursive=True):
        dst = os.path.join(CKPT_DIR, os.path.basename(src_p))
        if not os.path.exists(dst):
            shutil.copy2(src_p, dst)
            RESTORED['ckpt'] += 1
    if RESTORED['ckpt']:
        print(f"restored {RESTORED['ckpt']} checkpoint(s) into {CKPT_DIR}")

if os.path.exists(REPEAT_CSV):
    import pandas as _pd
    _df = _pd.read_csv(REPEAT_CSV)
    _df = _df[_pd.to_numeric(_df.psnr, errors='coerce').notna()]
    print('\ncompleted runs already on record:')
    for tag, grp in _df.groupby('tag'):
        print(f"  {tag:<14} seeds {sorted(int(s) for s in grp.seed)} "
              f"| PSNR {', '.join(f'{p:.3f}' for p in grp.psnr)}")
    print('\nthese will be SKIPPED below; everything else trains normally.')
else:
    print('no previous seed_repeats.csv found — this is a fresh run'
          + (' (add the previous notebook output as an Input to resume)' if ON_KAGGLE else ''))

no previous seed_repeats.csv found — this is a fresh run (add the previous notebook output as an Input to resume)


## 2. Cube split — held FIXED across every seed

In [5]:
train_cubes, val_cubes, holdout_cubes = split_cubes(data_dir=DATA_DIR, n_holdout=3,
                                                     val_fraction=0.2, seed=SEED)
print('\nsplit is seeded separately and never varies per training seed, so the spread '
      'measured below is training variance on a fixed split.')

CUBE-LEVEL SPLIT (grouped by RunID — no channel-level leakage)
  data_dir          : /kaggle/input/datasets/krishanyadav333/line-emission-data/Line Emission Data
  total cubes       : 14  across 11 distinct RunIDs
  seed=42  n_holdout=3  val_fraction=0.2
----------------------------------------------------------------------
  TRAIN   :  7 cubes | RunIDs ['0006', '0010', '0020', '0022', '0030', '0035']
  VAL     :  2 cubes | RunIDs ['0016', '0036']
  HOLDOUT :  5 cubes | RunIDs ['0002', '0025', '0026']  <-- inference only, NEVER trained/validated
----------------------------------------------------------------------
  HOLDOUT cube folders (reserved for moment-map evaluation):
    - run_0002_00560_rt_00
    - run_0002_00560_rt_01
    - run_0002_00560_rt_04
    - run_0025_01000_rt_04
    - run_0026_00005_rt_04

split is seeded separately and never varies per training seed, so the spread measured below is training variance on a fixed split.


## 3. Datasets — one plain pair, one augmented train set

`val_ds` is shared by every run and has augmentation **off**: an augmented validation set makes
the early-stopping metric non-deterministic and the runs incomparable.

In [6]:
common = dict(n_samples=N_SAMPLES, target_size=TARGET_SIZE, seed=SEED,
              subtract_continuum=SUBTRACT_CONTINUUM, continuum_n=CONTINUUM_N)
train_ds     = FITSChannelDataset(train_cubes, **common)
train_ds_aug = FITSChannelDataset(train_cubes, augment=True, **common)
val_ds       = FITSChannelDataset(val_cubes,   **common)
print('train:', len(train_ds), '| train(aug):', len(train_ds_aug), '| val:', len(val_ds))

# sanity: augmentation must vary the train item and leave val deterministic
torch.manual_seed(0)
n_orient = len({train_ds_aug[0][0].numpy().tobytes() for _ in range(40)})
val_fixed = len({val_ds[0][0].numpy().tobytes() for _ in range(5)}) == 1
print(f'augmented train item takes {n_orient} distinct orientations | val deterministic: {val_fixed}')
assert n_orient > 1 and val_fixed, 'augmentation wiring is wrong -- stop and fix before training'

[FITSChannelDataset] 7 cubes x ~150 channels = 1050 items | target 256x256 | continuum-subtracted (mean of first/last 5 channels)
[FITSChannelDataset] 7 cubes x ~150 channels = 1050 items | target 256x256 | continuum-subtracted (mean of first/last 5 channels) | D4 augmentation ON
[FITSChannelDataset] 2 cubes x ~150 channels = 300 items | target 256x256 | continuum-subtracted (mean of first/last 5 channels)
train: 1050 | train(aug): 1050 | val: 300
augmented train item takes 8 distinct orientations | val deterministic: True


## 3b. Rebuild `seed_repeats.csv` from checkpoints, if it is missing

Resume keys off the CSV, because that is where PSNR/SSIM/MSE live — a checkpoint stores only
`epoch` and `val_loss`. So checkpoints **without** the CSV would still trigger a full retrain.

They do not have to. Re-scoring a saved checkpoint on the validation split takes about a minute;
retraining the arm takes 30–180. This cell finds any `*_seed*.pth` with no matching CSV row,
rebuilds the model from the checkpoint's own metadata, and re-runs the same `val_metrics`
function the training loop used, so the recovered numbers are directly comparable.

`epochs_run` and `wall_time_s` cannot be recovered — they were never stored — and are left
blank rather than guessed. Nothing downstream depends on them except the §6b duration
diagnostic, which simply skips rows it cannot read.

No-op when the CSV is already present.

In [7]:
from src.training.sweep import val_metrics
from src.training.architectures import build_model

def _rows_in_csv(path):
    if not os.path.exists(path): return set()
    import csv as _csv
    with open(path, newline='') as f:
        return {(r['tag'], int(r['seed'])) for r in _csv.DictReader(f)
                if r.get('seed', '').strip().isdigit() and r.get('psnr')}

have = _rows_in_csv(REPEAT_CSV)
ckpts = sorted(glob.glob(os.path.join(CKPT_DIR, '*_seed*.pth')))
missing = []
for p in ckpts:
    stem = os.path.basename(p)[:-4]                 # e.g. winner_aug_seed43
    tag, _, seed_s = stem.rpartition('_seed')
    if seed_s.isdigit() and (tag, int(seed_s)) not in have:
        missing.append((p, tag, int(seed_s)))

if not missing:
    print(f'{len(ckpts)} checkpoint(s) on disk, {len(have)} CSV row(s) — nothing to rebuild')
else:
    print(f'{len(missing)} checkpoint(s) have no CSV row; re-scoring instead of retraining:')
    fields = ['tag','seed','best_epoch','epochs_run','best_val_loss','psnr','ssim','mse','wall_time_s']
    new_file = not os.path.exists(REPEAT_CSV)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=16, shuffle=False,
                                             num_workers=NW)
    with open(REPEAT_CSV, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=fields)
        if new_file: w.writeheader()
        for p, tag, seed in missing:
            ck = torch.load(p, map_location=device, weights_only=False)
            arch = ck.get('arch_key', 'unet')
            net = build_model(arch, base_channels=ck['base_channels'],
                              channel_multipliers=tuple(ck.get('channel_multipliers') or (1,2,4)),
                              use_beam=bool(ck.get('use_beam')),
                              latent_dim=ck.get('latent_dim', 128)).to(device)
            net.load_state_dict(ck['model_state_dict']); net.eval()
            m = val_metrics(net, val_loader, device, use_beam=bool(ck.get('use_beam')), arch=arch)
            w.writerow({'tag': tag, 'seed': seed, 'best_epoch': ck.get('epoch'),
                        'epochs_run': '', 'best_val_loss': ck.get('val_loss'),
                        'psnr': m['psnr'], 'ssim': m['ssim'], 'mse': m['mse'],
                        'wall_time_s': ''})
            f.flush()
            print(f"  {tag:<14} seed {seed}: epoch {ck.get('epoch')} | "
                  f"PSNR {m['psnr']:.4f} | SSIM {m['ssim']:.4f}  (re-scored)")
            del net
            if torch.cuda.is_available(): torch.cuda.empty_cache()
    print('\nCSV rebuilt — these arms will now be SKIPPED by resume.')

0 checkpoint(s) on disk, 0 CSV row(s) — nothing to rebuild


## 4. The three configurations

**The schedule is the change.** v4 ran `min_epochs=20, max_epochs=60, patience=5` and both of
its ceiling-limited runs (`v12` seed 42, `winner_p10` seed 44, each 60/60) were the best in
their arm. A run that is still improving when it is stopped does not report its configuration's
quality, it reports where the ceiling was. `max_epochs=200, patience=15` removes both cuts, so
every arm here trains to actual convergence and is compared at convergence.

That matters more than it sounds. In v4 the augmented arm ran a mean 38.7 epochs against plain
`winner`'s 27.3 — augmentation bought +11 epochs before early stopping fired. So `winner_aug`
(+1.322 dB) and `winner_p10` (+1.298 dB) posted near-identical gains, and both differed from
`winner` in the same way: more training. **Under a schedule neither arm can be truncated by,
that confound is gone** and `base_aug` − `base` measures augmentation alone.

`base` is v4's `v12` arm — the reference architecture and loss, now converged. It is a
*re-measurement* of the reference under this notebook's schedule, not a claim to reproduce
V12's checkpoint, and its PSNR is not comparable to V12's published 32.95 dB (different
`N_SAMPLES`). The moment scores are comparable: they are computed on full holdout cubes.

`wide_aug` is the only 48-channel arm kept. v4 could not distinguish that architecture from
`base` at n=3 (+0.376 dB, combined spread 1.344), so it is carried at n=5 and with augmentation
— matched to `base_aug` on everything except width and depth — to settle it rather than leave
it open. It is also v4's best-scoring arm, so it is the most likely final model.

Every arm is renamed off v4's tags on purpose. `repeat_config` resumes by `(tag, seed)`, so a
reused name would let section 1b skip seeds that were trained on the old schedule and quietly
mix two schedules inside one arm. v4's tags were `v12` / `winner` / `winner_aug` / `winner_p10`;
this run uses `base` / `base_aug` / `wide_aug`, which collide with none of them.

Dropped from v4: the un-augmented `winner` arm (the architecture question is now asked between
the two augmented arms) and `winner_p10` (patience is no longer a variable — every arm gets 15).


In [8]:
# One schedule for every arm: patience is no longer a variable being tested, it is a
# setting chosen from v4's result. max_epochs 60 -> 200 because 60 was BINDING in v4 (two
# runs finished 60/60 and were the best in their arm), and patience 5 -> 15 because v4's
# patience-10 arm gained +1.298 dB over patience-5 while halving the seed spread.
SCHED = dict(min_epochs=30, max_epochs=200, patience=15)

_base = dict(base_channels=32, channel_multipliers=(1, 2, 4),
             lr=1e-3, alpha=0.8, sched_patience=5, use_beam=False,
             batch_size=32, **SCHED)
_w48  = dict(base_channels=48, channel_multipliers=(1, 2, 4, 8),
             lr=0.0008196504330730313, alpha=0.8877681051398497,
             sched_patience=8, use_beam=False, batch_size=16, **SCHED)

# insertion order is the report order: augmentation pair first, then the architecture arm.
# 'wide_aug', NOT 'winner_aug': repeat_config resumes by (tag, seed), and v4's CSV already
# holds winner_aug at seeds 42/43/44 from the patience-5 / 60-epoch schedule. Reusing the
# name would make section 1b silently skip three of this arm's five seeds and mix two
# schedules inside one arm -- the exact confound this run exists to remove.
CONFIGS = {'base': _base, 'base_aug': dict(_base), 'wide_aug': _w48}

DATASETS = {'base': train_ds, 'base_aug': train_ds_aug, 'wide_aug': train_ds_aug}

for name, cfg in CONFIGS.items():
    aug = ' [augmented]' if DATASETS[name] is train_ds_aug else ''
    print(f"{name:<12} patience={cfg['patience']:<3} base={cfg['base_channels']:<3} "
          f"mult={cfg['channel_multipliers']}{aug}")
print(f'\n{len(CONFIGS)} configs x {N_SEEDS} seeds = {len(CONFIGS)*N_SEEDS} training runs')
print(f"schedule: min {SCHED['min_epochs']} / max {SCHED['max_epochs']} epochs, "
      f"patience {SCHED['patience']}")


base         patience=15  base=32  mult=(1, 2, 4)
base_aug     patience=15  base=32  mult=(1, 2, 4) [augmented]
wide_aug     patience=15  base=48  mult=(1, 2, 4, 8) [augmented]

3 configs x 5 seeds = 15 training runs
schedule: min 30 / max 200 epochs, patience 15


## 5. Train — 3 configs x 5 seeds

15 runs, roughly 19 h GPU at v4's measured throughput, so plan on 2–3 sessions. Rows append to
the CSV and each seed's best checkpoint is saved **into the Output** as it finishes, so a
session timeout costs the seed in flight and nothing else — re-attach this version's Output and
section 1b skips everything already on record. Keeping every seed's checkpoint (not just the
best) is what lets the moment-map protocol run on a chosen seed rather than whatever model
happened to be last in memory.

In [9]:
results = {}
t_all = time.time()

for name, cfg in CONFIGS.items():
    t0 = time.time()
    results[name] = repeat_config(
        DATASETS[name], val_ds, device,
        n_seeds=N_SEEDS, base_seed=SEED, out_csv=REPEAT_CSV, ckpt_dir=CKPT_DIR,
        tag=name, num_workers=NW, resume=True, verbose=True, **cfg)
    print(f'--- {name} finished in {(time.time()-t0)/60:.1f} min ---\n', flush=True)

print(f'all {len(CONFIGS)*N_SEEDS} runs in {(time.time()-t_all)/60:.1f} min -> {REPEAT_CSV}')

=== repeating 'base' over 5 seeds (split held fixed; only training seed varies) ===

--- base: seed 42 (1/5) ---
  ep   1 | train 0.0925 | val 0.0247 | lr 1.0e-03 (42s) *best
  ep   2 | train 0.0137 | val 0.0115 | lr 1.0e-03 (35s) *best
  ep   3 | train 0.0089 | val 0.0086 | lr 1.0e-03 (38s) *best
  ep   4 | train 0.0081 | val 0.0070 | lr 1.0e-03 (40s) *best
  ep   5 | train 0.0247 | val 0.0120 | lr 1.0e-03 (38s)
  ep   6 | train 0.0072 | val 0.0063 | lr 1.0e-03 (39s) *best
  ep   7 | train 0.0072 | val 0.0082 | lr 1.0e-03 (38s)
  ep   8 | train 0.0063 | val 0.0056 | lr 1.0e-03 (38s) *best
  ep   9 | train 0.0049 | val 0.0049 | lr 1.0e-03 (39s) *best
  ep  10 | train 0.0043 | val 0.0051 | lr 1.0e-03 (38s)
  ep  11 | train 0.0037 | val 0.0036 | lr 1.0e-03 (38s) *best
  ep  12 | train 0.0031 | val 0.0037 | lr 1.0e-03 (38s)
  ep  13 | train 0.0029 | val 0.0030 | lr 1.0e-03 (38s) *best
  ep  14 | train 0.0026 | val 0.0031 | lr 1.0e-03 (38s)
  ep  15 | train 0.0026 | val 0.0031 | lr 1.0e-03

## 6. Channel-level comparison, with error bars

In [ ]:
print('=' * 88)
print('{:<14} {:>22} {:>20} {:>22}'.format('config', 'PSNR (dB)', 'SSIM', 'MSE'))
print('-' * 88)
for name in CONFIGS:
    r = results[name]
    print('{:<14} {:>13.4f} +/- {:<6.4f} {:>11.4f} +/- {:<6.4f} {:>12.6f} +/- {:<8.6f}'.format(
        name, r['psnr']['mean'], r['psnr']['std'],
        r['ssim']['mean'], r['ssim']['std'], r['mse']['mean'], r['mse']['std']))
print('{:<14} {:>13.4f} {:>7} {:>11.4f} {:>7} {:>12.6f}'.format(
    'V12 pub n_s=50', V12['psnr'], '(n=1)', V12['ssim'], '(n=1)', V12['mse']))
print('=' * 88)

def verdict(a, b, name_a, name_b):
    """Compare two repeated configs, refusing to call a gap real if the spreads overlap."""
    ma, sa = a['psnr']['mean'], a['psnr']['std']
    mb, sb = b['psnr']['mean'], b['psnr']['std']
    gap = ma - mb
    pooled = float(np.sqrt(sa ** 2 + sb ** 2)) or 1e-12
    print(f'\n{name_a} - {name_b}: {gap:+.3f} dB  (spreads {sa:.3f} / {sb:.3f}, '
          f'combined {pooled:.3f})')
    if abs(gap) > 2 * pooled:
        print(f'  -> gap exceeds 2x the combined spread: treat as REAL')
    elif abs(gap) > pooled:
        print(f'  -> gap is 1-2x the combined spread: SUGGESTIVE, not established at n={N_SEEDS}')
    else:
        print(f'  -> gap is within the combined spread: INDISTINGUISHABLE from seed noise')
    return gap, pooled

# the augmentation question, with duration held fixed by the shared schedule
verdict(results['base_aug'], results['base'], 'base_aug', 'base')
# the architecture question, both arms augmented and on the same schedule
verdict(results['wide_aug'], results['base_aug'], 'wide_aug', 'base_aug')

print(f'\nNOTE: with n={N_SEEDS} the std is still a noisy estimate. These thresholds are a '
      'guard against over-claiming, not a significance test.')

## 6b. Did the arms converge, or did the ceiling cut them off again?

v4's headline mechanism was duration: `epochs_run` correlated **+0.721** with PSNR across its 12
runs, replicating the sweep's +0.650, and its two ceiling-limited runs were the best in their
arm. This run raised `max_epochs` to 200 to remove that cut.

So the check has changed. It is no longer "does longer help" — that is settled and already spent.
It is **"did 200 epochs turn out to be enough"**. Any run that finishes at `epochs_run == 200`
was still improving when it stopped, which means this run has the same defect v4 had and the
ceiling needs raising again before its numbers can be called converged.

The `epochs_run` vs PSNR correlation is still reported, because it needs no extra training and it
is the direct evidence for whether duration is still doing work at this schedule. If the arms
have genuinely converged, that correlation should now be **weak** — the runs that trained longer
should no longer be systematically the better ones.


In [ ]:
import math

def pearson(x, y):
    n = len(x)
    if n < 2: return float('nan')
    mx, my = sum(x)/n, sum(y)/n
    cov = sum((a-mx)*(b-my) for a, b in zip(x, y))
    sx = math.sqrt(sum((a-mx)**2 for a in x)); sy = math.sqrt(sum((b-my)**2 for b in y))
    return cov/(sx*sy) if sx > 0 and sy > 0 else float('nan')

print('epochs_run vs PSNR, per config (does a longer run score better?)')
print('-' * 74)
all_ep, all_ps = [], []
for name in CONFIGS:
    ep = [r['epochs_run'] for r in results[name]['rows']]
    ps = [r['psnr'] for r in results[name]['rows']]
    all_ep += ep; all_ps += ps
    print('{:<12} epochs {:<22} PSNR {:<26} r={:+.3f}'.format(
        name, ', '.join(str(int(e)) for e in ep),
        ', '.join(f'{p:.2f}' for p in ps), pearson(ep, ps)))
print('-' * 74)
print('{:<12} pooled over all {} runs{:>34}r={:+.3f}'.format(
    'ALL', len(all_ep), '', pearson(all_ep, all_ps)))
print('  (sweep reference: r = +0.650 over its 12 runs)')

print('  (v4 reference: r = +0.721 over its 12 runs, at max_epochs=60)')

# Did the ceiling bind again? A run that stops at max_epochs was still improving.
MAX_EP = SCHED['max_epochs']
pegged = [(name, int(r['seed']), int(r['epochs_run']))
          for name in CONFIGS for r in results[name]['rows']
          if int(r['epochs_run']) >= MAX_EP]
print(f'\nconvergence check -- runs that hit the {MAX_EP}-epoch ceiling:')
if pegged:
    for name, seed, ep in pegged:
        print(f'  {name} seed {seed}: {ep}/{MAX_EP} epochs -- STILL IMPROVING when stopped')
    print(f'  -> {len(pegged)}/{len(all_ep)} runs truncated. Same defect as v4: raise '
          f'max_epochs and re-run before calling these numbers converged.')
else:
    print(f'  none -- every run early-stopped before {MAX_EP}. The arms converged and the '
          f'comparison is at convergence, which is what v4 could not claim.')

for name in CONFIGS:
    ep = [r['epochs_run'] for r in results[name]['rows']]
    print(f'  {name:<12} epochs min {int(min(ep))} / mean {np.mean(ep):.1f} / '
          f'max {int(max(ep))}')

## 7. Loss curves — all seeds, all configs

In [ ]:
import pandas as pd
df = pd.read_csv(REPEAT_CSV)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
for ax, metric in zip(axes, ['psnr', 'ssim', 'mse']):
    for i, name in enumerate(CONFIGS):
        sub = df[df.tag == name]
        ax.scatter([i] * len(sub), sub[metric], s=60, alpha=0.8, zorder=5,
                   label=name if metric == 'psnr' else None)
        ax.errorbar(i, sub[metric].mean(), yerr=sub[metric].std(ddof=1),
                    fmt='_', ms=28, capsize=8, color='#333', zorder=4)
    if metric == 'psnr':
        ax.axhline(V12['psnr'], ls='--', color='#E8715A', lw=1.2, label='V12 published (n=1, N_SAMPLES=50)')
        ax.legend(fontsize=8)
    ax.set_xticks(range(len(CONFIGS))); ax.set_xticklabels(list(CONFIGS), rotation=20, ha='right')
    ax.set_title(metric.upper()); ax.grid(axis='y', alpha=0.3)
fig.suptitle(f'Per-seed spread across {N_SEEDS} seeds (dots = seeds, bars = mean +/- std)',
             fontweight='bold')
plt.tight_layout()
p = os.path.join(OUT_DIR, 'seed_spread.png'); plt.savefig(p, dpi=140); plt.show()
print('saved ->', p)

## 8. Moment-map protocol on the best seed of each config

The scientific deliverable. Runs the same all-5-holdout evaluation V12 was judged on. The beam
A/B is the cautionary precedent: PSNR rose while M0 fell from +69.8% to +59.8% with doubled
variance, so a config is not promoted on pixel metrics alone.

In [ ]:
import bettermoments as bm
from astropy.io import fits
from src.evaluation.moment_maps import (generate_moment_maps,
                                        moment_improvement)
from src.evaluation.classical import summarise_improvements

BS = 32
moments = ['M0', 'M1', 'M2']

def mdiff(a, b):
    m = np.isfinite(a) & np.isfinite(b)
    return float(np.nanmean(np.abs(a[m] - b[m])))

def load_net(ckpt_path):
    ck = torch.load(ckpt_path, map_location=device, weights_only=False)
    net = UNet(in_channels=1, out_channels=1, base_channels=ck['base_channels'],
               channel_multipliers=ck['channel_multipliers'], time_emb_dim=128,
               num_res_blocks=2, groups=math.gcd(8, ck['base_channels']),
               beam_dim=ck.get('beam_dim', 0)).to(device)
    net.load_state_dict(ck['model_state_dict']); net.eval()
    return net, ck

def denoise_cube_unet(ho, net):
    with fits.open(ho['dirty'], memmap=False) as h:
        raw = np.ascontiguousarray(h[0].data).astype(np.float32)
    csub = raw - continuum_of(raw, CONTINUUM_N)[None]
    C, H, W = csub.shape
    lo = csub.reshape(C, -1).min(axis=1); hi = csub.reshape(C, -1).max(axis=1)
    rng_ = hi - lo; nz = rng_ > 0
    norm = np.zeros_like(csub)
    norm[nz] = (csub[nz] - lo[nz, None, None]) / rng_[nz, None, None]
    out = np.empty_like(csub)
    with torch.no_grad():
        for s in range(0, C, BS):
            t = torch.from_numpy(norm[s:s+BS])[:, None].to(device)
            t256 = F.interpolate(t, (TARGET_SIZE, TARGET_SIZE), mode='bilinear', align_corners=False)
            tz = torch.zeros(t256.size(0), dtype=torch.long, device=device)
            pred = net(t256, tz)
            back = F.interpolate(pred, (H, W), mode='bilinear', align_corners=False)[:, 0].cpu().numpy()
            for k in range(back.shape[0]):
                ch = s + k
                out[ch] = back[k] * rng_[ch] + lo[ch] if rng_[ch] > 0 else np.full((H, W), lo[ch], np.float32)
    return out, csub

# cache clean/dirty moment maps once; reused by all three configs
cache = {}
for ho in holdout_cubes:
    with fits.open(ho['clean'], memmap=False) as h:
        craw = np.ascontiguousarray(h[0].data).astype(np.float32)
    ccsub = craw - continuum_of(craw, CONTINUUM_N)[None]
    with fits.open(ho['dirty'], memmap=False) as h:
        draw = np.ascontiguousarray(h[0].data).astype(np.float32)
    dcsub = draw - continuum_of(draw, CONTINUUM_N)[None]
    _, velax = bm.load_cube(ho['dirty'])
    cache[ho['folder']] = {'velax': velax,
                           'clean': generate_moment_maps(None, data_velax=(ccsub, velax)),
                           'dirty': generate_moment_maps(None, data_velax=(dcsub, velax))}
print('cached clean/dirty moment maps for', len(cache), 'cubes')

moment_summaries, moment_rows = {}, {}
for name in CONFIGS:
    best_seed = int(max(results[name]['rows'], key=lambda r: r['psnr'])['seed'])
    ckpt = os.path.join(CKPT_DIR, f'{name}_seed{best_seed}.pth')
    net, ck = load_net(ckpt)
    print(f'\n=== {name} (best seed {best_seed}, epoch {ck.get("epoch")}) ===', flush=True)
    rows = []
    for ho in holdout_cubes:
        den, _ = denoise_cube_unet(ho, net)
        e = cache[ho['folder']]
        no = generate_moment_maps(None, data_velax=(den, e['velax']))
        row = {'cube': ho['folder'], 'seed': best_seed}
        # Scored over the signal mask: averaging over empty sky let pixels with no line
        # dominate, which hit M2 hardest (its denominator vanishes exactly there). The mask
        # comes from the CLEAN M0 alone, so it is identical for every method and cannot be
        # tuned per model; `_all` keeps the unmasked value beside it.
        _imp = moment_improvement(e['clean'], e['dirty'], no)
        for nm in moments:
            row['imp_' + nm] = round(_imp[nm], 2)
            row['imp_' + nm + '_all'] = round(_imp[nm + '_all'], 2)
        rows.append(row)
        print('  {:<24} M0 {:>7.1f}%  M1 {:>7.1f}%  M2 {:>7.1f}%'.format(
            ho['folder'], row['imp_M0'], row['imp_M1'], row['imp_M2']))
    moment_rows[name] = rows
    moment_summaries[name] = summarise_improvements(rows)
    s = moment_summaries[name]
    print('  -> ' + '  '.join(f'{m} {s[m]["mean"]:+.1f}%+/-{s[m]["std"]:.1f}' for m in moments))
    del net
    if torch.cuda.is_available(): torch.cuda.empty_cache()

## 9. Final table — the midterm's decisive comparison

In [ ]:
print('=' * 96)
print('{:<20} {:>14} {:>18} {:>18} {:>18}'.format('config', 'PSNR (dB)', 'M0 (%)', 'M1 (%)', 'M2 (%)'))
print('-' * 96)
for name in CONFIGS:
    s, r = moment_summaries[name], results[name]
    print('{:<20} {:>8.2f}+/-{:<4.2f} {:>10.1f} +/-{:<5.1f} {:>10.1f} +/-{:<5.1f} {:>10.1f} +/-{:<5.1f}'.format(
        name, r['psnr']['mean'], r['psnr']['std'],
        s['M0']['mean'], s['M0']['std'], s['M1']['mean'], s['M1']['std'],
        s['M2']['mean'], s['M2']['std']))
print('{:<20} {:>8.2f}{:>6} {:>10.1f} +/-{:<5.1f} {:>10.1f} +/-{:<5.1f} {:>10.1f} +/-{:<5.1f}'.format(
    'V12 pub UNCLIPPED', V12['psnr'], '(n=1)',
    V12['M0'][0], V12['M0'][1], V12['M1'][0], V12['M1'][1], V12['M2'][0], V12['M2'][1]))
print('{:<20} {:>8} {:>6} {:>10.1f} +/-{:<5.1f} {:>10.1f} +/-{:<5.1f} {:>10.1f} +/-{:<5.1f}'.format(
    'beam pub UNCLIPPED', '34.94', '(n=1)', 59.8, 33.0, 19.2, 8.4, 23.2, 20.2))
print('=' * 96)
print('RULES.md #6 -- the two "pub" rows are on the UNCLIPPED whole-map moment metric')
print('  (05 v12 / v15, before bab16d0). This run is signal-masked and noise-clipped.')
print('  Their M0/M1/M2 may NOT be differenced against the arms above: on the same models,')
print('  08 v2 read M0 +82.0 unclipped where v4 read +27.7 clipped. The gap is the metric.')
print('  PSNR differs too -- those rows were measured at N_SAMPLES=50, this run uses 150.')

print('\nPromotion check — a config is only promoted if it wins on the SCIENTIFIC metric:')
ref = moment_summaries['base']
for name in ('base_aug', 'wide_aug'):
    s = moment_summaries[name]
    print(f'\n  {name} vs base (this notebook, matched schedule and duration):')
    for m in moments:
        gap = s[m]['mean'] - ref[m]['mean']
        pooled = float(np.sqrt(s[m]['std'] ** 2 + ref[m]['std'] ** 2)) or 1e-12
        flag = 'REAL' if abs(gap) > 2 * pooled else ('suggestive' if abs(gap) > pooled else 'within noise')
        print(f'    {m}: {gap:+6.1f} pp  (cube-to-cube spread {pooled:.1f} pp) -> {flag}')

csv_path = os.path.join(OUT_DIR, 'seed_moment_summary.csv')
with open(csv_path, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['config', 'seed', 'cube', 'imp_M0', 'imp_M1', 'imp_M2'])
    for name, rows in moment_rows.items():
        for r in rows:
            w.writerow([name, r['seed'], r['cube'], r['imp_M0'], r['imp_M1'], r['imp_M2']])
        s = moment_summaries[name]
        w.writerow([name, '', 'MEAN'] + [round(s[m]['mean'], 2) for m in moments])
        w.writerow([name, '', 'STD'] + [round(s[m]['std'], 2) for m in moments])
print('\nsaved ->', csv_path)

## 10. Artifact diagnostics — did augmentation reduce invented structure?

In [ ]:
from src.evaluation.artifacts import channel_artifacts, summarise

# The hallucination artifact was attributed to small data, which is exactly what
# augmentation is supposed to mitigate. This measures it rather than assuming it.
art = {}
for name in CONFIGS:
    best_seed = int(max(results[name]['rows'], key=lambda r: r['psnr'])['seed'])
    net, _ = load_net(os.path.join(CKPT_DIR, f'{name}_seed{best_seed}.pth'))
    rows = []
    with torch.no_grad():
        for i in range(len(val_ds)):
            d, c = val_ds[i]
            pred = net(d[None].to(device), torch.zeros(1, dtype=torch.long, device=device))
            cl = c[0].numpy()
            if cl.max() <= 0:
                continue
            rows.append(channel_artifacts(cl, d[0].numpy(), pred[0, 0].cpu().numpy()))
    art[name] = summarise(rows)
    a = art[name]
    print(f'{name:<12} overshoot {a["overshoot_mean"]:.3f} | floor leak {a["floor_leak_mean"]:+.5f} '
          f'| channels with invented blob {a["frac_channels_with_blob"]:.1%} '
          f'| blobs/channel {a["blobs_per_channel"]:.3f}')
    if 'low_snr_blobs_per_channel' in a:
        print(f'{"":12} low-SNR blobs/ch {a["low_snr_blobs_per_channel"]:.3f} vs '
              f'high-SNR {a["high_snr_blobs_per_channel"]:.3f} (split at SNR {a["snr_median"]:.1f})')
    del net
    if torch.cuda.is_available(): torch.cuda.empty_cache()

blob_plain = art['base']['frac_channels_with_blob']
blob_aug   = art['base_aug']['frac_channels_with_blob']
print(f'\naugmentation effect on invented structure: {blob_plain:.1%} -> {blob_aug:.1%} '
      f'of channels ({blob_aug - blob_plain:+.1%} pp)')
print(f"  (v4, confounded with duration: 37.7% -> 29.7%, -8.0 pp)")

art_csv = os.path.join(OUT_DIR, 'artifact_diagnostics_seeds.csv')
keys = sorted({k for a in art.values() for k in a})
with open(art_csv, 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['config'] + keys)
    for name, a in art.items():
        w.writerow([name] + [a.get(k, '') for k in keys])
print('saved ->', art_csv)

## 11. Persist artifacts to /kaggle/working — backstop

Checkpoints and `seed_repeats.csv` are already there: section 1 points `CKPT_DIR` and
`REPEAT_CSV` at `/kaggle/working` so they land in the Output as training produces them
(RULES.md #1). This cell catches what is written later — the moment and artifact CSVs, the
figures — and re-reports the files already in place. It must never be the only save.

In [ ]:
import shutil
if ON_KAGGLE:
    # every per-seed checkpoint, not just the best: a resumed session needs the
    # completed arms' checkpoints to run their moment maps without retraining
    import glob as _glob
    wanted = [REPEAT_CSV, csv_path, art_csv,
              os.path.join(OUT_DIR, 'seed_spread.png')]
    wanted += sorted(_glob.glob(os.path.join(CKPT_DIR, '*_seed*.pth')))
    for p in wanted:
        if not os.path.exists(p):
            continue
        dst = '/kaggle/working/' + os.path.basename(p)
        if os.path.abspath(p) == os.path.abspath(dst):
            # already written straight to the Output by section 2 (RULES.md #1)
            print('already persisted ->', os.path.basename(p),
                  f'({os.path.getsize(p)/1e6:.1f} MB)')
            continue
        shutil.copy2(p, dst)
        print('persisted ->', os.path.basename(p),
              f'({os.path.getsize(p)/1e6:.1f} MB)')
    print('\ndownload the CSVs from the kernel Output tab and commit them to results/')
else:
    print('not on Kaggle — nothing to persist')